# Model Training & Comparison

Six models are trained and compared on the combined MSE metric:

| Model | Type |
|---|---|
| OLS | Linear baseline |
| LASSO | Regularised linear |
| Ridge | Regularised linear |
| ElasticNet | Regularised linear |
| Decision Tree | Non-linear baseline |
| Random Forest | Ensemble tree |
| Neural Network (MLP) | Deep learning |
| XGBoost | Gradient boosting |
| LightGBM | Gradient boosting |

**Final model selected: XGBoost** (lowest combined MSE, best balance of accuracy and interpretability)

> Hyperparameters for NN and XGBoost were found via `GridSearchCV` (tuning cells are archived at the bottom).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from preprocessing import prepare_data
from models import (
    evaluate, train_linear_models, train_decision_tree,
    train_random_forest, train_neural_network, train_xgboost, train_lightgbm
)

df_raw = pd.read_csv('kinder_clean.csv')

(
    X_train, X_test,
    y_read_train, y_read_test,
    y_math_train, y_math_test,
    preprocessor, num_cols, cat_cols
) = prepare_data(df_raw)

print('Data ready.')

## Linear Models (OLS, LASSO, Ridge, ElasticNet)

In [ ]:
linear_models = train_linear_models(preprocessor, X_train, y_read_train, y_math_train)

linear_results = []
for name, (pipe_r, pipe_m) in linear_models.items():
    metrics = evaluate(pipe_r, pipe_m, X_test, y_read_test, y_math_test)
    linear_results.append({'model': name, **metrics})

pd.DataFrame(linear_results).set_index('model').round(4)

### Linear Model Feature Importances (coefficients)

In [ ]:
for name, (pipe_r, pipe_m) in linear_models.items():
    feat_names = [f.split('__')[-1] for f in pipe_r.named_steps['prep'].get_feature_names_out()]
    coefs = pipe_r.named_steps['model'].coef_
    df_fi = pd.DataFrame({'feature': feat_names, 'coef': coefs, 'abs': np.abs(coefs)})
    df_fi = df_fi.sort_values('abs', ascending=False).head(10)

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.barh(df_fi['feature'][::-1], df_fi['coef'][::-1])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{name} — Top 10 Coefficients (Reading)')
    plt.tight_layout()
    plt.show()

## Decision Tree

In [ ]:
pipe_read_dt, pipe_math_dt = train_decision_tree(preprocessor, X_train, y_read_train, y_math_train)
dt_metrics = evaluate(pipe_read_dt, pipe_math_dt, X_test, y_read_test, y_math_test)
print('Decision Tree:', dt_metrics)

## Random Forest

In [ ]:
pipe_read_rf, pipe_math_rf = train_random_forest(preprocessor, X_train, y_read_train, y_math_train)
rf_metrics = evaluate(pipe_read_rf, pipe_math_rf, X_test, y_read_test, y_math_test)
print('Random Forest:', rf_metrics)

In [ ]:
feat_names = [n.split('__')[-1] for n in pipe_read_rf.named_steps['prep'].get_feature_names_out()]
rf_imp = pd.DataFrame({'feature': feat_names,
                        'importance': pipe_read_rf.named_steps['model'].feature_importances_})
rf_imp = rf_imp.sort_values('importance', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(rf_imp['feature'][::-1], rf_imp['importance'][::-1], color='forestgreen')
ax.set_title('Random Forest — Top 10 Feature Importances (Reading)')
plt.tight_layout()
plt.show()

## Neural Network (MLP)

In [ ]:
pipe_read_nn, pipe_math_nn = train_neural_network(preprocessor, X_train, y_read_train, y_math_train)
nn_metrics = evaluate(pipe_read_nn, pipe_math_nn, X_test, y_read_test, y_math_test)
print('Neural Network:', nn_metrics)

## XGBoost (Final Model)

In [ ]:
pipe_read_xgb, pipe_math_xgb = train_xgboost(preprocessor, X_train, y_read_train, y_math_train)
xgb_metrics = evaluate(pipe_read_xgb, pipe_math_xgb, X_test, y_read_test, y_math_test)
print('XGBoost:', xgb_metrics)

## LightGBM

In [ ]:
pipe_read_lgb, pipe_math_lgb = train_lightgbm(preprocessor, X_train, y_read_train, y_math_train)
lgb_metrics = evaluate(pipe_read_lgb, pipe_math_lgb, X_test, y_read_test, y_math_test)
print('LightGBM:', lgb_metrics)

## Model Comparison Summary

In [ ]:
all_results = []
for name, metrics in [
    *[(k, evaluate(v[0], v[1], X_test, y_read_test, y_math_test)) for k, v in linear_models.items()],
    ('Decision Tree', dt_metrics),
    ('Random Forest', rf_metrics),
    ('Neural Network', nn_metrics),
    ('XGBoost', xgb_metrics),
    ('LightGBM', lgb_metrics),
]:
    all_results.append({'model': name, **metrics})

results_df = pd.DataFrame(all_results).set_index('model').round(4).sort_values('mse_o')
print(results_df)

fig, ax = plt.subplots(figsize=(9, 4))
results_df['mse_o'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Combined MSE_o by Model (lower is better)')
ax.set_ylabel('MSE_o')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## Appendix: GridSearchCV Tuning (archived, not run in main pipeline)

These cells document how the best hyperparameters for NN and XGBoost were found.
They are kept for reproducibility but are not executed in normal runs.

In [ ]:
# ── XGBoost tuning (archived) ─────────────────────────────────────────────
# from sklearn.model_selection import GridSearchCV
# from xgboost import XGBRegressor
#
# param_grid = {
#     'model__n_estimators': [100, 200, 300],
#     'model__max_depth': [3, 5, 7],
#     'model__learning_rate': [0.01, 0.05, 0.1],
#     'model__subsample': [0.8, 1.0],
#     'model__colsample_bytree': [0.8, 1.0],
#     'model__gamma': [0, 0.1, 0.3],
#     'model__reg_lambda': [1, 5, 10],
#     'model__reg_alpha': [0, 0.1, 0.5],
# }
# ...
# Best READ : n_estimators=300, max_depth=3, lr=0.1, subsample=0.8, gamma=0.1, reg_alpha=0
# Best MATH : n_estimators=300, max_depth=3, lr=0.1, subsample=0.8, gamma=0.0, reg_alpha=0.5
pass

In [ ]:
# ── NN tuning (archived) ──────────────────────────────────────────────────
# param_grid = {
#     'model__hidden_layer_sizes': [(32,16), (64,32), (128,64)],
#     'model__alpha': [0.0001, 0.0005, 0.001, 0.005],
#     'model__learning_rate_init': [0.0005, 0.001, 0.003],
#     'model__activation': ['relu'],
#     'model__solver': ['adam'],
# }
# Best READ : hidden=(64,32), alpha=0.0001, lr=0.003
# Best MATH : hidden=(64,32), alpha=0.001,  lr=0.003
pass